# V20 – Aufgaben: Musterlösungen

Vollständige Lösungen zu den fünf Aufgaben aus [../03_aufgaben.ipynb](../03_aufgaben.ipynb).

## 📦 Voraussetzungen
- Die Aufgaben selbst zuerst versucht.


In [1]:
import sqlite3

def setup_db() -> sqlite3.Connection:
    """Legt eine frische In-Memory-DB mit deterministischen Daten an."""
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE Maschinen (
            Maschinen_ID INTEGER PRIMARY KEY,
            Name         TEXT NOT NULL,
            Typ          TEXT NOT NULL,
            Baujahr      INTEGER NOT NULL
        )
    """)
    cur.execute("""
        CREATE TABLE Wartungen (
            Wartung_ID    INTEGER PRIMARY KEY,
            Maschinen_ID  INTEGER NOT NULL,
            Wartungstyp   TEXT NOT NULL,
            Datum         TEXT NOT NULL,
            Kosten        REAL NOT NULL,
            FOREIGN KEY (Maschinen_ID) REFERENCES Maschinen(Maschinen_ID)
        )
    """)
    maschinen = [
        (1, "CNC-Fräse 01",    "Fräse",    2018),
        (2, "CNC-Fräse 02",    "Fräse",    2020),
        (3, "Drehmaschine 01", "Drehbank", 2016),
        (4, "Drehmaschine 02", "Drehbank", 2021),
        (5, "Roboter-Arm R1",  "Roboter",  2019),
    ]
    wartungen = [
        (1, 1, "Inspektion",   "2024-01-10",  150.00),
        (2, 1, "Reparatur",    "2024-03-05", 1200.00),
        (3, 2, "Inspektion",   "2024-02-14",  180.00),
        (4, 3, "Ölwechsel",    "2024-01-20",   90.00),
        (5, 3, "Reparatur",    "2024-04-12", 2500.00),
        (6, 3, "Inspektion",   "2024-05-18",  200.00),
        (7, 4, "Inspektion",   "2024-02-28",  160.00),
        (8, 5, "Kalibrierung", "2024-03-22",  450.00),
    ]
    cur.executemany("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", maschinen)
    cur.executemany("INSERT INTO Wartungen VALUES (?, ?, ?, ?, ?)", wartungen)
    conn.commit()
    return conn

# Kurztest
conn = setup_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM Maschinen")
print("Maschinen:", cur.fetchone()[0])
cur.execute("SELECT COUNT(*) FROM Wartungen")
print("Wartungen:", cur.fetchone()[0])
conn.close()


Maschinen: 5
Wartungen: 8


## Lösung A1 – `COUNT`

In [2]:
def anzahl_wartungen(cursor):
    cursor.execute("SELECT COUNT(*) FROM Wartungen")
    return cursor.fetchone()[0]

conn = setup_db(); cur = conn.cursor()
assert anzahl_wartungen(cur) == 8
print("A1:", anzahl_wartungen(cur))
conn.close()


A1: 8


## Lösung A2 – `SUM`

In [3]:
def gesamtkosten(cursor):
    cursor.execute("SELECT SUM(Kosten) FROM Wartungen")
    return cursor.fetchone()[0]

conn = setup_db(); cur = conn.cursor()
assert gesamtkosten(cur) == 4930.0
print("A2:", gesamtkosten(cur))
conn.close()


A2: 4930.0


## Lösung A3 – `GROUP BY` + `COUNT`

In [4]:
def wartungen_pro_typ(cursor):
    cursor.execute("""
        SELECT Wartungstyp, COUNT(*)
        FROM Wartungen
        GROUP BY Wartungstyp
        ORDER BY Wartungstyp
    """)
    return cursor.fetchall()

conn = setup_db(); cur = conn.cursor()
erwartet = [("Inspektion", 4), ("Kalibrierung", 1), ("Reparatur", 2), ("Ölwechsel", 1)]
assert wartungen_pro_typ(cur) == erwartet
for zeile in wartungen_pro_typ(cur):
    print(zeile)
conn.close()


('Inspektion', 4)
('Kalibrierung', 1)
('Reparatur', 2)
('Ölwechsel', 1)


## Lösung A4 – `INNER JOIN` + `GROUP BY` + `ORDER BY`

Die Abfrage kombiniert alle in V20 eingeführten Bausteine: JOIN, Gruppierung, Aggregat, Sortierung, `LIMIT`. Das Ergebnis ist ein einzelnes Tupel `(Name, Gesamtkosten)`.

In [5]:
def top_kosten_maschine(cursor):
    cursor.execute("""
        SELECT M.Name, SUM(W.Kosten) AS gesamt
        FROM Maschinen M
        INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
        GROUP BY M.Maschinen_ID
        ORDER BY gesamt DESC
        LIMIT 1
    """)
    return cursor.fetchone()

conn = setup_db(); cur = conn.cursor()
ergebnis = top_kosten_maschine(cur)
assert ergebnis == ("Drehmaschine 01", 2790.0)
print("A4:", ergebnis)
conn.close()


A4: ('Drehmaschine 01', 2790.0)


## Lösung A5 – `LEFT JOIN` + Filter auf `NULL`

Erst die 6. Maschine einfügen, dann den LEFT JOIN mit dem klassischen `IS NULL`-Trick absetzen.

In [6]:
def maschinen_ohne_wartung(cursor):
    cursor.execute("""
        SELECT M.Name
        FROM Maschinen M
        LEFT JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
        WHERE W.Wartung_ID IS NULL
        ORDER BY M.Name
    """)
    return [zeile[0] for zeile in cursor.fetchall()]

conn = setup_db(); cur = conn.cursor()
cur.execute("INSERT INTO Maschinen VALUES (?, ?, ?, ?)",
            (6, "Roboter-Arm R2", "Roboter", 2023))

ergebnis = maschinen_ohne_wartung(cur)
assert ergebnis == ["Roboter-Arm R2"]
print("A5:", ergebnis)
conn.close()


A5: ['Roboter-Arm R2']


## ✅ Alles gelöst

Gut gemacht! Mit diesen Mustern (JOIN, GROUP BY, HAVING, LEFT JOIN + IS NULL) deckst du schon einen großen Teil der täglichen SQL-Praxis ab.
